# Final held-out before/after evaluation

Run after model/prompt choices are fixed. This is inference only. It evaluates original and selected-adapter models on the prescribed eight test patients, with the same FP16/NF4 protocol and no tuning. The original dataset bytes are embedded; no upload widget. Download final_test_results.zip and return it for the final report.


In [ ]:
import base64, hashlib, io, json, subprocess, sys, zipfile
from pathlib import Path
FINAL_PROJECT = Path("/content/trialgpt_final_evaluation")
PAYLOAD = '...'
EXPECTED = {'evaluate_final.py': '91d6f8e3d6c9808e91f801ca8c31eec27fd5a587adc607c143ac274921d6e6bc', 'test_final_evaluation.py': '36ea0d9c7112f7ca8d1212964198919e5463415f65d48a632dae650b9d743d47', 'evaluate_adapter.py': '93471485cbede95c0289c8028bc2391da0dbcba8e096d8350b4e41ced30ceac2', 'baseline.py': '65b95fb88c7e7e8aafa6962f384cbae1eabd559f058240977062c5b71a808bd3', 'prepare_model_inputs.py': '004b35f7e65d70e64a1e6b8fa75e6f00e865dee309d9b6d1be0beae59642683d', 'eda.py': '23806970afd89f37e2f954592ed06946c03a23e20384b801f1c4b672a17747e5', 'configs/model_input_config.json': '8da817effb1afaa454f112d2de141890a75dbebdfb1ed0d678b3e3be59d84b02', 'configs/tokenizer_lock.json': '9a26905bf50ef272621deeefc55d16728452ffde5032735001be2e2be851aaaf', 'data/splits/split_manifest.json': '945d010de9941f26564b5ebd2d8bf20148581e410c483721d6f27ce39f67a8d5', 'requirements-colab-finetuning.txt': '6fec25f70d7099b1df3fe2228e931461b56b92df543c6382ab5fff8c0bc0c3d9', 'requirements-colab-baseline.txt': 'fed534126c21757c0f430c3be93694b3963c9308ec40c26de8b7696af4c2b6ed', 'requirements-model-inputs.txt': '89f561ddae7a4c9fdaac73dcc4cb2bfaa9957039e5ab1caff3693f172b23b066', 'requirements-data.txt': '1069eda3d380e8122956223bd414f829e3e2dc934e25c395a707d31094b63c7b', 'train-00000-of-00001.parquet': '14c547184fac9b013cd17441fd028845b40150c76b022e58334f5494077e5234'}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for name, digest in EXPECTED.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError("Embedded checksum differs: " + name)
        destination = FINAL_PROJECT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
def final_run(arguments):
    with subprocess.Popen(arguments, cwd=FINAL_PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout: print(line, end="", flush=True)
        code = process.wait()
    if code: raise RuntimeError("Final evaluation failed; read the output above. Partial predictions are preserved.")
print("Restored final-evaluation code and original dataset with exact checksums; no test predictions made yet.")


## Install pinned dependencies (needed in a fresh runtime)


In [ ]:
final_run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
final_run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(FINAL_PROJECT / "requirements-colab-finetuning.txt")])


## Run the fixed final comparison


In [ ]:
import torch
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime. This evaluates both models; no training occurs.")
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
candidates = []
if globals().get("TRAINING") is not None:
    candidates.append(Path(TRAINING))
candidates += [Path("/content/drive/MyDrive/TrialGPT_Assignment2/selected_adapter"),
               Path("/content/drive/MyDrive/TrialGPT_Assignment2/qlora_v1")]
FINAL_TRAINING = next((p for p in candidates if (p / "training_manifest.json").is_file()
                       and (p / "adapter/adapter_model.safetensors").is_file()), None)
if FINAL_TRAINING is None:
    raise FileNotFoundError("No selected adapter found. Keep TRAINING from the speculative notebook or save the adapter to Drive.")
FINAL_RUN = Path("/content/drive/MyDrive/TrialGPT_Assignment2/final_test_v1")
final_run([sys.executable, "-m", "unittest", "-v", "test_final_evaluation"])
final_run([sys.executable, "-u", "evaluate_final.py", "--training-output", str(FINAL_TRAINING), "--output", str(FINAL_RUN)])
print((FINAL_RUN / "summary.md").read_text())


## Download results


In [ ]:
from google.colab import files
download = Path("/content/final_test_results.zip")
with zipfile.ZipFile(download, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in FINAL_RUN.rglob("*"):
        if path.is_file(): archive.write(path, path.relative_to(FINAL_RUN).as_posix())
files.download(str(download))
